# Algoritmos de Regresión

In [1]:
from pyspark.sql import SparkSession

# 1. Iniciamos Spark en este nuevo notebook
spark = SparkSession.builder.appName("Semana12_Supervisados").getOrCreate()

# 2. Leemos DIRECTAMENTE los datos que guardamos en el Paso 1
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 4. Mostramos la tabla para comprobar que TODO está ahí
df_clusters.select("marca", "precio_kg", "rating", "prediction").show(10)

+-----+-----------------+-----------------+----------+
|marca|        precio_kg|           rating|prediction|
+-----+-----------------+-----------------+----------+
|    0|9.020000457763672|4.800000190734863|         1|
|    1|6.860000133514404|4.699999809265137|         0|
|    3|5.150000095367432|4.699999809265137|         0|
|    1|5.349999904632568|4.699999809265137|         0|
|    0|7.639999866485596|4.800000190734863|         1|
|    1|9.359999656677246|4.900000095367432|         0|
|    3|4.670000076293945|4.800000190734863|         0|
|    0|7.510000228881836|4.800000190734863|         1|
|    3|4.670000076293945|4.699999809265137|         0|
|    0|8.739999771118164|4.800000190734863|         1|
+-----+-----------------+-----------------+----------+
only showing top 10 rows



In [6]:
# 1. Creamos el VectorAssembler (sin los precios)
assembler_regresion = VectorAssembler(
    inputCols=["rating", "opiniones"], 
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_clusters)

# 2. Escalamos las características
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# 3. Renombramos la variable objetivo Y
df_para_regresion = df_para_regresion.withColumnRenamed("precio_kg", "label_precio")

# =======================================================================
# Borramos la columna prediction del K-Means 
# para que no choque con la de la Regresión Lineal
df_para_regresion = df_para_regresion.drop("prediction")
# =======================================================================

# 4. Dividimos en Entrenamiento (70%) y Prueba (30%)
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

In [7]:
# Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# Entrenar el modelo con los datos de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# Hacer las predicciones sobre los datos de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# Mostrar las predicciones junto al precio real
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------------+-----------------+
|marca|      label_precio|       prediction|
+-----+------------------+-----------------+
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
+-----+------------------+-----------------+
only showing top 10 rows



In [8]:
# Configurar los evaluadores de regresión
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       ")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Error promedio del modelo):  {rmse:.4f}")
print("==================================================")

     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       
R² (Coeficiente de Determinación): 5.68%
RMSE (Error promedio del modelo):  2.8363


In [9]:
# Imprimir la intersección (b) y los coeficientes (m)
print(f"Intersección (Precio base): {lr_reg_model.intercept:.4f}")
print(f"Coeficiente de 'rating':    {lr_reg_model.coefficients[0]:.4f}")
print(f"Coeficiente de 'opiniones': {lr_reg_model.coefficients[1]:.4f}")

Intersección (Precio base): -14.6993
Coeficiente de 'rating':    0.6410
Coeficiente de 'opiniones': 0.0421
